### Import Check ⬇️

In [1]:
# Import Check
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Alle Pakete erfolgreich geladen ✅")

Alle Pakete erfolgreich geladen ✅


### ROM-Struktur verstehen, welche Version ist vorhanden? ⬇️

In [2]:
ROM_PATH = "../data/pokemon_firered.gba"

with open(ROM_PATH, "rb") as f:
    header = f.read(0xC0)

game_code = header[0xAC:0xB0].decode("ascii")
game_title = header[0xA0:0xAC].decode("ascii").strip()

print(f"Spieltitel: {game_title}")
print(f"Game Code:  {game_code}")

Spieltitel: POKEMON FIRE
Game Code:  BPRD


### Pokemon Zeichenkodierung Hexa nicht Standard: Quelle -> Claude.ai ⬇️

In [3]:
CHAR_MAP = {
    0xBB: "A", 0xBC: "B", 0xBD: "C", 0xBE: "D", 0xBF: "E",
    0xC0: "F", 0xC1: "G", 0xC2: "H", 0xC3: "I", 0xC4: "J",
    0xC5: "K", 0xC6: "L", 0xC7: "M", 0xC8: "N", 0xC9: "O",
    0xCA: "P", 0xCB: "Q", 0xCC: "R", 0xCD: "S", 0xCE: "T",
    0xCF: "U", 0xD0: "V", 0xD1: "W", 0xD2: "X", 0xD3: "Y",
    0xD4: "Z",
    0xD5: "a", 0xD6: "b", 0xD7: "c", 0xD8: "d", 0xD9: "e",
    0xDA: "f", 0xDB: "g", 0xDC: "h", 0xDD: "i", 0xDE: "j",
    0xDF: "k", 0xE0: "l", 0xE1: "m", 0xE2: "n", 0xE3: "o",
    0xE4: "p", 0xE5: "q", 0xE6: "r", 0xE7: "s", 0xE8: "t",
    0xE9: "u", 0xEA: "v", 0xEB: "w", 0xEC: "x", 0xED: "y",
    0xEE: "z",
    0x00: " ", 0xFF: ""  # Leerzeichen & String-Ende
}

### Offset Testausgabe aller Pokemon mit 10 Byte Abstand ⬇️

In [4]:
def decode_pokemon_string(data):
    result = ""
    for byte in data:
        if byte == 0xFF:  # String-Ende
            break
        result += CHAR_MAP.get(byte, "?")
    return result.strip()

# Anpassbare Variablen festlegen und Namen auslesen
NAMES_OFFSET = 0x245EE0
POKEMON_COUNT = 151
NAME_LENGTH = 10

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

for p in pokemon_namen:
    print(f"#{p['id']:03d} {p['name']}")

#001 SANDAM
#002 ER
#003 AN?
#004 RINA
#005 OQUEEN
#006 DORAN?
#007 IDORINO
#008 NIDOKING
#009 PIEPI
#010 PIXI
#011 VULPIX
#012 VULNON
#013 A
#014 LUFF
#015 DELUFF
#016 AT
#017 LBAT
#018 YRAPLA
#019 DUFLOR
#020 GIFLOR
#021 PARAS
#022 PARASEK
#023 
#024 
#025 DIGD
#026 A
#027 DRI
#028 UZI
#029 NOBILIKAT
#030 ENTON
#031 ENTORON
#032 MENKI
#033 RASAFF
#034 FUKANO
#035 
#036 I
#037 SEL
#038 PUTZI
#039 APPO
#040 BRA
#041 KADABRA
#042 SIMSALA
#043 MACHOLLO
#044 
#045 K
#046 EI
#047 NSA
#048 IGARIA
#049 ZENIA
#050 NTACHA
#051 ENTOXA
#052 KLEINSTEIN
#053 
#054 GEOWAZ
#055 PONITA
#056 GALLOP
#057 A
#058 ON
#059 US
#060 NETILO
#061 GNETON
#062 ORENTA
#063 DODU
#064 DODRI
#065 JUROB
#066 JUGONG
#067 SLEIMA
#068 
#069 OK
#070 HAS
#071 TOS
#072 BULAK
#073 LPOLLO
#074 GENGAR
#075 ONIX
#076 TRAUMATO
#077 
#078 KRABBY
#079 
#080 ER
#081 OBAL
#082 TROBAL
#083 EI
#084 OKOWEI
#085 TRAGOSSO
#086 KNOGGA
#087 KICKLEE
#088 NOCKCHA
#089 N
#090 P
#091 N
#092 MOG
#093 ORN
#094 ZEROS
#095 HANEIRA
#096 TANGELA
#0

### Offensichtlich falscher Offset in vorheriger Zelle
### Suche nach "BISASAM" (Pokemon 001)
### In Pokemon-Kodierung übersetzen ⬇️

In [5]:
CHAR_MAP_REVERSE = {v: k for k, v in CHAR_MAP.items()}

def encode_pokemon_string(text):
    return bytes([CHAR_MAP_REVERSE.get(c, 0x00) for c in text])

# Nach "BISASAM" in der ROM suchen
search = encode_pokemon_string("BISASAM")
print(f"Suche nach Byte: {search.hex()}")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

position = rom_data.find(search)
print(f"Gefunden an Position: {hex(position)}")

# Neuer Offset = Position von BISASAM
# (da BISASAM Pokémon #001 ist, ist das direkt der Basis-Offset)
print(f"Neuer NAMES_OFFSET: {hex(position)}")

Suche nach Byte: bcc3cdbbcdbbc7
Gefunden an Position: 0x1907b7
Neuer NAMES_OFFSET: 0x1907b7


### Neuen Offset testen ⬇️

In [6]:
NAMES_OFFSET = 0x1907b7
POKEMON_COUNT = 151
NAME_LENGTH = 10

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

for p in pokemon_namen:
    print(f"#{p['id']:03d} {p['name']}")

#001 BISASAM so
#002 ll es sein
#003 ??Es ist l
#004 eicht zu t
#005 rainieren?
#006 ???? du m?
#007 chtest als
#008 o das?PFLA
#009 NZEN?POK?M
#010 ON BISASAM
#011 ?
#012 OK?MON ist
#013 wirklich?
#014 energiegel
#015 aden?
#016 rh?lt????
#017 PROF? EICH
#018 ? Wenn du
#019 auf ein wi
#020 ldes?POK?M
#021 ON triffst
#022 ? dann kan
#023 n dein?POK
#024 ?MON dageg
#025 en k?mpfen
#026 ?
#027 CH? ??? we
#028 nn du dein
#029 ?POK?MON k
#030 ?mpfen l?s
#031 st? wird e
#032 s?st?rker?
#033 
#034 H? Hallo?
#035 ????Wie ge
#036 ht es dem
#037 POK?MON??E
#038 s scheint
#039 dich sehr
#040 zu m?gen??
#041 Du musst a
#042 ls POK?MON
#043 TRAINER s
#044 ehr?talent
#045 iert sein?
#046 ?Du hast e
#047 twas f?r m
#048 ich?
#049 ergibt?PRO
#050 F? EICH da
#051 s PAKET?
#052 h? Auf die
#053 sen SPEZIA
#054 L?POK?BALL
#055 ?warte ich
#056 schon lan
#057 ge? ?Viele
#058 n Dank?
#059 OF? EICH?
#060 Ach ja? Ic
#061 h habe ein
#062 e?gro?e Au
#063 fgabe f?r
#064 euch?
#065 dem Tisch
#066 dort seht
#067 i

### Falscher Offset mit "BISASAM", wahrscheinlich einfach nur Storytext
### Suche eingrenzen mit "BISASAM" und "BISAKNOSP" ⬇️

In [7]:
search1 = encode_pokemon_string("BISASAM")
search2 = encode_pokemon_string("BISAKNOSP")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

# Alle Vorkommen von BISASAM finden
pos = 0
while True:
    position = rom_data.find(search1, pos)
    if position == -1:
        break

    # Prüfen ob BISAKNOSP genau 10 Bytes später kommt
    next_pos = position + 10
    if rom_data[next_pos:next_pos + len(search2)] == search2:
        print(f"✅ Namenstabelle gefunden bei: {hex(position)}")
        break
    else:
        print(f"❌ Falscher Treffer bei: {hex(position)}")

    pos = position + 1

❌ Falscher Treffer bei: 0x1907b7
❌ Falscher Treffer bei: 0x190814
❌ Falscher Treffer bei: 0x245dbb


### Falsche Treffer in Zelle zuvor.
### Prüfen was 10 Byte später steht.
### Vielleicht doch anderer Byte Absatnd? ⬇️

In [8]:
search1 = encode_pokemon_string("BISASAM")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(search1, pos)
    if position == -1:
        break

    # Zeige was nach BISASAM kommt (20 Bytes)
    after = rom_data[position:position + 30]
    decoded = decode_pokemon_string(after)

    print(f"Position {hex(position)}: '{decoded}'")
    print(f"  Raw Bytes: {after.hex()}")
    print()

    pos = position + 1

Position 0x1907b7: 'BISASAM soll es sein??Es ist l'
  Raw Bytes: bcc3cdbbcdbbc700e7e3e0e000d9e700e7d9dde2abfebfe700dde7e800e0

Position 0x190814: 'BISASAM?'
  Raw Bytes: bcc3cdbbcdbbc7acffbeddd9e7d9e700cac9c51bc7c9c800dde7e800ebdd

Position 0x245dbb: 'BISASAM'
  Raw Bytes: bcc3cdbbcdbbc7ff000000bcc3cdbbc5c8c9cdcaff00bcc3cdbbc0c6c9cc



### Position 0x245dbb: 'BISASAM' sieht vielversprechend aus ff = String Ende dann 3 Nullbyte Abstand
### bcc3cdbbcdbbc7ff000000 = (B I S A S A M = 7 Byte) + (ff = 1 Byte) + (00 00 00 = 3 Byte) also 7+1+3=11 Byte
### Gegentest: bcc3cdbbc5c8c9cdcaff00 = (B I S A K N O S P = 9 Byte) + (ff = 1 Byte) + (00 = 1 Byte) also 9+1+1=11 Byte
### Es sind nicht 10 Byte Abstand sondern 11
### Offset Test mit neuen Erkenntnissen ⬇️

In [9]:
NAMES_OFFSET = 0x245dbb
POKEMON_COUNT = 151
NAME_LENGTH = 11  # ← geändert!

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

df_namen = pd.DataFrame(pokemon_namen)
df_namen

,id,name
0,1,BISASAM
1,2,BISAKNOSP
2,3,BISAFLOR
3,4,GLUMANDA
4,5,GLUTEXO
...,...,...
146,147,DRATINI
147,148,DRAGONIR
148,149,DRAGORAN
149,150,MEWTU


### Offset für Pokenamen richtig identifiziert
### Jetzt Basiswerte also HP, ATK, DEF, Speed, SP.ATK, SP.DEF (45, 49, 49, 45, 65, 65) extrahieren ⬇️


In [10]:
import struct
bisasam_stats = bytes([45, 49, 49, 45, 65, 65])

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(bisasam_stats, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    print(f"Raw Bytes: {rom_data[position:position+28].hex()}")
    print()

    pos = position + 1

Treffer bei: 0x2546c4
Raw Bytes: 2d31312d41410c032d400001000000001f1446030107410000030000

Nichts weiter gefunden!


### Mit aller Wahrscheinlichkeit Treffer
### 2d 31 31 2d 41 41 = sind Bisasams Werte (45 49 49 45 65 65)
### Alle stats bei allen Pokemon auslesen ⬇️

In [14]:
STATS_OFFSET = 0x2546c4
POKEMON_COUNT = 151

pokemon_stats = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = STATS_OFFSET + (i * 28)
        f.seek(offset)
        data = f.read(28)

        stats = struct.unpack_from("BBBBBB", data, 0)

        pokemon_stats.append({
            "hp":             stats[0],
            "angriff":        stats[1],
            "verteidigung":   stats[2],
            "speed":          stats[3],
            "sp_angriff":     stats[4],
            "sp_verteidigung":stats[5],
        })

df_stats = pd.DataFrame(pokemon_stats)
pd.set_option('display.max_rows', None)  # Alle Zeilen anzeigen
df_stats

,hp,angriff,verteidigung,speed,sp_angriff,sp_verteidigung
0,45,49,49,45,65,65
1,60,62,63,60,80,80
2,80,82,83,80,100,100
3,39,52,43,65,60,50
4,58,64,58,80,80,65
5,78,84,78,100,109,85
6,44,48,65,43,50,64
7,59,63,80,58,65,80
8,79,83,100,78,85,105
9,45,30,35,45,20,20


In [12]:
# Typ-ID
TYPES = {
    0:  "Normal",  1:  "Kampf",   2:  "Flug",
    3:  "Gift",    4:  "Boden",   5:  "Gestein",
    6:  "Käfer",   7:  "Geist",   8:  "Stahl",
    10: "Feuer",   11: "Wasser",  12: "Gras",
    13: "Elektro", 14: "Psycho",  15: "Eis",
    16: "Drache",  17: "Unlicht"
}

pokemon_typen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = STATS_OFFSET + (i * 28)
        f.seek(offset)
        data = f.read(28)

        stats = struct.unpack_from("BBBBBBBB", data, 0)

        pokemon_typen.append({
            "typ1": TYPES.get(stats[6], f"Unbekannt({stats[6]})"),
            "typ2": TYPES.get(stats[7], f"Unbekannt({stats[7]})")
        })

df_typen = pd.DataFrame(pokemon_typen)
df_typen

,typ1,typ2
0,Gras,Gift
1,Gras,Gift
2,Gras,Gift
3,Feuer,Feuer
4,Feuer,Feuer
5,Feuer,Flug
6,Wasser,Wasser
7,Wasser,Wasser
8,Wasser,Wasser
9,Käfer,Käfer


### Stats ausgelesen
### Typen sollten im selben Byte-Block sein direkt hinter den stats (Check = Bisasam 001/00 sollte Gras/Gift sein) ⬇️

### Typen wurden erfolgreich extrahiert
### Jetzt Zusammenführung von drei df zu einem df (pd.concat weil alle df gleich viele Zeilen, Reihenfolge) ⬇️

In [15]:
df_pokemon = pd.concat([df_namen, df_stats, df_typen], axis=1)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_pokemon = df_pokemon.set_index("id")
df_pokemon

,name,hp,angriff,verteidigung,speed,sp_angriff,sp_verteidigung,typ1,typ2
id,,,,,,,,,
1,BISASAM,45,49,49,45,65,65,Gras,Gift
2,BISAKNOSP,60,62,63,60,80,80,Gras,Gift
3,BISAFLOR,80,82,83,80,100,100,Gras,Gift
4,GLUMANDA,39,52,43,65,60,50,Feuer,Feuer
5,GLUTEXO,58,64,58,80,80,65,Feuer,Feuer
6,GLURAK,78,84,78,100,109,85,Feuer,Flug
7,SCHIGGY,44,48,65,43,50,64,Wasser,Wasser
8,SCHILLOK,59,63,80,58,65,80,Wasser,Wasser
9,TURTOK,79,83,100,78,85,105,Wasser,Wasser
